# 🤖 Treino, Validação e Otimização

- Divisão **70/15/15 estratificada e agrupada por `md5`** (nenhum ficheiro aparece em dois conjuntos).
- 7 classificadores comparados no Validation Set e com **Stratified Group 5-Fold CV** no treino.
- Métrica principal: **Recall** da classe Malware, mas só entre modelos com CV F1 ≥ 0,95 (senão o Naive Bayes, que classifica tudo como malware, ganhava).
- `GridSearchCV` do XGBoost orientado ao Recall; em caso de empate mantém-se o baseline.

In [ ]:
import sys, os, warnings
sys.path.append('../src')
warnings.filterwarnings('ignore')
import pandas as pd
from features import prepare_datasets
from models import (split_dataset, evaluate_all_models, tune_xgboost, choose_baseline_or_tuned,
                    overfitting_report, save_model)

df = pd.read_csv('../data/processed/ransom_cleaned.csv')
X, y, le, groups = prepare_datasets(df, return_groups=True)
print(X.shape, list(le.classes_))
X_train, X_val, X_test, y_train, y_val, y_test, g_train = split_dataset(X, y, groups)

In [ ]:
tabela, ajustados = evaluate_all_models(X_train, y_train, X_val, y_val, g_train)
tabela.round(4)

In [ ]:
melhor = tabela.iloc[0]['Model']
print('Melhor modelo:', melhor)
otimizado, params, resultados_grid = tune_xgboost(X_train, y_train, g_train)
modelo_final, versao = choose_baseline_or_tuned(ajustados['XGBoost'], otimizado, X_val, y_val)
print('Modelo final:', versao)

In [ ]:
ajustados['XGBoost (otimizado)'] = otimizado
overfitting_report(ajustados, X_train, y_train, X_val, y_val).round(4)

O modelo só é guardado aqui; o **Test Set não foi usado** até agora (é avaliado uma única vez no notebook 04).

In [ ]:
save_model(modelo_final, X.columns, '../models', meta={'modelo': melhor, 'versao': versao})
print('✅ Modelo guardado em ../models/ransomware_model.pkl')